## Sample wise consolidated data

In [ ]:
library(glue)
library(stringr)
library(dplyr)
library(R.matlab)

create_consolidated_data <- function(sample){
            # main_dir <- glue("../../pipeline_testing/specialist_generalist_with_cluster/{sample}")
            main_dir <- glue("../../pipeline_testing/specialist_generalist/{sample}")
            
            target_files <- list.files(path = main_dir, pattern = "*.tsv$", full.names = TRUE)
            
            sg_data_list <- lapply(target_files, read.table)
            
            filenames <- tools::file_path_sans_ext(basename(target_files))
            arc_names <- ifelse(grepl("generalists_cell", filenames, ignore.case = TRUE), "Generalist", gsub("_specialists.*", "", filenames))
            arc_names <- sub("Archetype_", "Archetype", arc_names, ignore.case = TRUE)
            
            sg_data_list <- Map(function(df, n) {
              df$archetype <- n
              return(df)
            }, sg_data_list, arc_names)
            
            sg_data_merged <- do.call(rbind, sg_data_list)
            
            rownames(sg_data_merged) <- NULL
            
            common_cell_data <- readLines(glue("../../pipeline_testing/specialist_generalist/{sample}/common_cells_between_archetypes_pairwise.txt"))
            
            if (length(common_cell_data)>0){
                cell_removed_before_mellon <- read.table(glue("../../pipeline_testing/specialist_generalist/{sample}/common_cells_between_archetypes_pairwise.txt"), sep="\t")
                names(cell_removed_before_mellon) <- 'x'
                cell_removed_before_mellon$archetype <- NA
                sg_data_merged <- rbind(sg_data_merged, cell_removed_before_mellon)
            }
            
            sample_main_meta_data <- read.table(glue("../../pipeline_testing/data/{sample}/{sample}_required_cell_modified.tsv"), sep="\t", header=T)
            
            sg_data_merged <- merge(sg_data_merged, sample_main_meta_data, by.x = "x", by.y = "index", all = TRUE)
            sg_data_merged <- sg_data_merged[,1:2]
            
            cons_full_df <- read.csv("../../pipeline_testing/pareto_front_consolidated_data.csv", row.names=1)
            cons_full_df$archetype <- sub("Archetype_", "Archetype", cons_full_df$archetype, ignore.case = TRUE)
            rownames(cons_full_df) <- cons_full_df$index
            sample_df <-cons_full_df[cons_full_df$Sample == sample,]

            arc_coord <- readMat(glue("../../pipeline_testing/parti_result/{sample}/Variables.mat"))
            arc_coord <- as.data.frame(arc_coord$arc)
            rownames(arc_coord) <- seq_len(nrow(arc_coord))
            colnames(arc_coord) <- paste0("PC", seq_len(ncol(arc_coord)))
            arc_no <- paste0("PC", 1:(nrow(arc_coord)-1))
            
            # arc_coord <- read.csv(glue("/home/subhasis/pipeline_testing/pareto_front_plots/{sample}/arc_coordinates.csv"), row.names=1)
            # arc_no <- paste0("PC", 1:(nrow(arc_coord)-1))
            sample_df <- sample_df[,c(arc_no, "Cell_type", "archetype")]
            
            sample_df <- sample_df %>% 
                         rename(Arch_after_mellon = archetype)
            
            sample_df$Arch_after_mellon <- gsub("Transitory_cells", NA, sample_df$Arch_after_mellon)
            sample_df$Arch_after_mellon <- gsub("generalists", "Generalist", sample_df$Arch_after_mellon)
            
            sample_df_mod <- merge(sg_data_merged, sample_df, by.x = "x", by.y = "row.names", all=TRUE)
            sample_df_mod <- sample_df_mod %>%
                              rename(Arch_before_mellon = archetype) %>%
                              rename(index = x)
            
            sample_df_mod  <- sample_df_mod [, c("index", arc_no, "Cell_type",	"Arch_before_mellon", "Arch_after_mellon")]
                
            
            transitory_data <- read.csv(glue("../../pipeline_testing/transitory_cells/{sample}/{sample}_archetype_contribution_score.csv"), row.names=1)
            
            collated_final <- merge(sample_df_mod, transitory_data, by.x = "index", by.y = "row.names", all=TRUE)
            
            required_col <- c('index', arc_no, 'Cell_type', 'Arch_before_mellon', 'Arch_after_mellon', 'cont_mod')
            
            collated_final <- collated_final %>% 
                                  select(all_of(required_col))
            
            collated_final <- collated_final %>% 
                                 rename(Arch_after_transitory = cont_mod)
            
            collated_final$Arch_after_transitory[collated_final$Arch_after_transitory == ""] <- NA
            collated_final$Arch_after_transitory <- ifelse(is.na(collated_final$Arch_after_transitory), collated_final$Arch_after_mellon, collated_final$Arch_after_transitory)
            
            collated_final <- collated_final %>%
                                  mutate(
                                    across(
                                      last_col(), 
                                      ~ ifelse(
                                          str_count(., "Archetype?") == 2,        
                                          str_replace_all(., "Archetype?", "Transitory"), 
                                          .                                      
                                      )
                                    )
                                  )
            
            collated_final <- collated_final %>%
                                      mutate(Arch_after_transitory = str_replace(Arch_after_transitory, "Transitory_(\\d+)_Transitory_(\\d+)", "Transitory_\\1_\\2"))
            
            cent_exp_data <- read.table(glue("../../pipeline_testing/data/{sample}/{sample}_gene_expression_parti_input.tsv"))
            cell_data <- read.csv(glue("../../pipeline_testing/data/{sample}/{sample}_required_cell_modified.tsv"), sep="\t",header=T,row.names=1)
            rownames(cent_exp_data) <-  rownames(cell_data)
            
            genes <- read.csv(glue("../../pipeline_testing/data/{sample}/genes.list"), header=FALSE)
            colnames(cent_exp_data) <- genes$V1
            
            csc_markers <- c('ABCG2', 'ALCAM', 'ALDH1A1', 'BMI1', 'CD200', 'CD24', 'CD44', 'CEACAM6', 'DCLK1', 'DPP4', 'FLOT2', 'ITGB1', 'LGR5', 'NANOG', 'POU5F1', 'PROM1', 'MSI1', 'SOX9')
            epi_markers <- c('MUC2', 'TFF1', 'EPCAM', 'KRT19', 'DCLK1', 'ATP1B3', 'CA1', 'CEACAM1', 'GUCA2A')
            myo_markers <- c('CD68', 'CD14', 'FCN1', 'CD11B')
            stromal_arkers <- c('TNC', 'VIM', 'COL1A1', 'TAGLN', 'SPARC', 'ACTA2', 'VWF', 'PECAM')
            t_markers <- c('CD3D', 'CD3E', 'NKG7', 'CCR6', 'KLRB1', 'CD8A')
            
            all_markers <- unique(c(csc_markers, epi_markers, myo_markers, stromal_arkers, t_markers))
            
            cent_exp_data <- cent_exp_data[,colnames(cent_exp_data) %in% all_markers]
            collated_final <- merge(collated_final, cent_exp_data, by.x = "index", by.y = "row.names", all=TRUE)
            
            collated_final <- collated_final %>% 
                                 rename(cell_type = Cell_type) 
            
            rownames(collated_final) <- collated_final$index
            collated_final <- subset(collated_final, select = -c(index))
            dir.create("../../pipeline_testing/consolidated_data_samplewise_test/", recursive=T)
            write.csv(collated_final, glue("../../pipeline_testing/consolidated_data_samplewise_test/{sample}_consolidated_data.csv"))

            return(paste("Saved:", {sample}))
        }

samples_list <- c("SMC02-T", "SMC05-T")

execution_log <- lapply(samples_list, create_consolidated_data)
print(execution_log)

In [ ]:
library(glue)
library(stringr)
library(dplyr)

sample <- "SMC05-T"
main_dir <- glue("../../pipeline_testing/specialist_generalist_with_cluster/{sample}")

target_files <- list.files(path = main_dir, pattern = "*.tsv$", full.names = TRUE)

sg_data_list <- lapply(target_files, read.table)

filenames <- tools::file_path_sans_ext(basename(target_files))
arc_names <- ifelse(grepl("generalists_cell", filenames, ignore.case = TRUE), "Generalist", gsub("_specialists.*", "", filenames))
arc_names <- sub("Archetype_", "Archetype", arc_names, ignore.case = TRUE)

sg_data_list <- Map(function(df, n) {
  df$archetype <- n
  return(df)
}, sg_data_list, arc_names)

sg_data_merged <- do.call(rbind, sg_data_list)

rownames(sg_data_merged) <- NULL

common_cell_data <- readLines(glue("../../pipeline_testing/specialist_generalist/{sample}/common_cells_between_archetypes_pairwise.txt"))

if (length(common_cell_data)>0){
    cell_removed_before_mellon <- read.table(glue("../../pipeline_testing/specialist_generalist/{sample}/common_cells_between_archetypes_pairwise.txt"), sep="\t")
    names(cell_removed_before_mellon) <- 'x'
    cell_removed_before_mellon$archetype <- NA
    sg_data_merged <- rbind(sg_data_merged, cell_removed_before_mellon)
}

sample_main_meta_data <- read.table(glue("../../pipeline_testing/data/{sample}/{sample}_required_cell_modified.tsv"), sep="\t", header=T)

sg_data_merged <- merge(sg_data_merged, sample_main_meta_data, by.x = "x", by.y = "index", all = TRUE)
sg_data_merged <- sg_data_merged[,1:2]

cons_full_df <- read.csv("../../pipeline_testing/pareto_front_consolidated_data.csv", row.names=1)
cons_full_df$archetype <- sub("Archetype_", "Archetype", cons_full_df$archetype, ignore.case = TRUE)
rownames(cons_full_df) <- cons_full_df$index
sample_df <-cons_full_df[cons_full_df$Sample == sample,]

arc_coord <- readMat(glue("../../pipeline_testing/parti_result/{sample}/Variables.mat"))
arc_coord <- as.data.frame(arc_coord$arc)
rownames(arc_coord) <- seq_len(nrow(arc_coord))
colnames(arc_coord) <- paste0("PC", seq_len(ncol(arc_coord)))
arc_no <- paste0("PC", 1:(nrow(arc_coord)-1))

# arc_coord <- read.csv(glue("/home/subhasis/pipeline_testing/pareto_front_plots/{sample}/arc_coordinates.csv"), row.names=1)
# arc_no <- paste0("PC", 1:(nrow(arc_coord)-1))

sample_df <- sample_df[,c(arc_no, "Cell_type", "archetype")]

sample_df <- sample_df %>% 
             rename(Arch_after_mellon = archetype)

sample_df$Arch_after_mellon <- gsub("Transitory_cells", NA, sample_df$Arch_after_mellon)
sample_df$Arch_after_mellon <- gsub("generalists", "Generalist", sample_df$Arch_after_mellon)

sample_df_mod <- merge(sg_data_merged, sample_df, by.x = "x", by.y = "row.names", all=TRUE)
sample_df_mod <- sample_df_mod %>%
                  rename(Arch_before_mellon = archetype) %>%
                  rename(index = x)

sample_df_mod  <- sample_df_mod [, c("index", arc_no, "Cell_type",	"Arch_before_mellon", "Arch_after_mellon")]
    

transitory_data <- read.csv(glue("../../pipeline_testing/transitory_cells/{sample}/{sample}_archetype_contribution_score.csv"), row.names=1)

collated_final <- merge(sample_df_mod, transitory_data, by.x = "index", by.y = "row.names", all=TRUE)

required_col <- c('index', arc_no, 'Cell_type', 'Arch_before_mellon', 'Arch_after_mellon', 'cont_mod')

collated_final <- collated_final %>% 
                      select(all_of(required_col))

collated_final <- collated_final %>% 
                     rename(Arch_after_transitory = cont_mod)

collated_final$Arch_after_transitory[collated_final$Arch_after_transitory == ""] <- NA
collated_final$Arch_after_transitory <- ifelse(is.na(collated_final$Arch_after_transitory), collated_final$Arch_after_mellon, collated_final$Arch_after_transitory)

collated_final <- collated_final %>%
                      mutate(
                        across(
                          last_col(), 
                          ~ ifelse(
                              str_count(., "Archetype?") == 2,        
                              str_replace_all(., "Archetype?", "Transitory"), 
                              .                                   
                          )
                        )
                      )

collated_final <- collated_final %>%
                          mutate(Arch_after_transitory = str_replace(Arch_after_transitory, "Transitory_(\\d+)_Transitory_(\\d+)", "Transitory_\\1_\\2"))

cent_exp_data <- read.table(glue("../../pipeline_testing/data/{sample}/{sample}_gene_expression_parti_input.tsv"))
cell_data <- read.csv(glue("../../pipeline_testing/data/{sample}/{sample}_required_cell_modified.tsv"), sep="\t",header=T,row.names=1)
rownames(cent_exp_data) <-  rownames(cell_data)

genes <- read.csv(glue("../../pipeline_testing/data/{sample}/genes.list"), header=FALSE)
colnames(cent_exp_data) <- genes$V1

csc_markers <- c('ABCG2', 'ALCAM', 'ALDH1A1', 'BMI1', 'CD200', 'CD24', 'CD44', 'CEACAM6', 'DCLK1', 'DPP4', 'FLOT2', 'ITGB1', 'LGR5', 'NANOG', 'POU5F1', 'PROM1', 'MSI1', 'SOX9')
epi_markers <- c('MUC2', 'TFF1', 'EPCAM', 'KRT19', 'DCLK1', 'ATP1B3', 'CA1', 'CEACAM1', 'GUCA2A')
myo_markers <- c('CD68', 'CD14', 'FCN1', 'CD11B')
stromal_arkers <- c('TNC', 'VIM', 'COL1A1', 'TAGLN', 'SPARC', 'ACTA2', 'VWF', 'PECAM')
t_markers <- c('CD3D', 'CD3E', 'NKG7', 'CCR6', 'KLRB1', 'CD8A')

all_markers <- unique(c(csc_markers, epi_markers, myo_markers, stromal_arkers, t_markers))

cent_exp_data <- cent_exp_data[,colnames(cent_exp_data) %in% all_markers]
collated_final <- merge(collated_final, cent_exp_data, by.x = "index", by.y = "row.names", all=TRUE)

collated_final <- collated_final %>% 
                     rename(cell_type = Cell_type) 

collated_final <- collated_final[!duplicated(collated_final$index),]
rownames(collated_final) <- collated_final$index
collated_final <- subset(collated_final, select = -c(index))
dir.create("../../pipeline_testing/consolidated_data_samplewise_test/", recursive=T)
write.csv(collated_final, glue("../../pipeline_testing/consolidated_data_samplewise_test/{sample}_consolidated_data.csv"))

## Reordering of rownames in consolidated data according to the colnames of TPM_mod or rownames of cell_metadata_file

In [ ]:
library(glue)
samples <- c('SMC01-N', 'SMC01-T', 'SMC02-T', 'SMC03-N', 'SMC03-T', 'SMC04-N', 'SMC04-T', 'SMC05-N', 'SMC05-T', 'SMC06-N', 'SMC06-T', 'SMC07-N', 'SMC07-T', 
             'SMC08-N', 'SMC08-T', 'SMC09-N', 'SMC09-T', 'SMC10-N', 'SMC10-T', 'SMC11-T', 'SMC14-T', 'SMC15-T', 'SMC16-T','SMC17-T', 'SMC18-T', 'SMC19-T', 
             'SMC20-T', 'SMC21-T', 'SMC22-T', 'SMC23-T', 'SMC24-T', 'SMC25-T')

# file_list <- list.files(path = "../../pipeline_testing/consolidated_data_samplewise/", pattern = "\\.csv$", full.names = TRUE)

for (i in 1:length(samples)){
  cons_data <- read.csv(glue("../../pipeline_testing/consolidated_data_samplewise_test/{samples[i]}_consolidated_data.csv"), row.names=1)
  tpm_data <- read.csv(glue("../../pipeline_testing/data/{samples[i]}/{samples[i]}_TPM_mod.csv"), row.names=1, check.names = FALSE)
  cells_metadata <- read.csv(glue("../../pipeline_testing/data/{samples[i]}/{samples[i]}_required_cell_modified.tsv"), sep="\t", row.names=1)
  cons_data_reordered <- cons_data[rownames(cells_metadata),]
  print(identical(colnames(tpm_data), rownames(cons_data_reordered)))
  write.csv(cons_data_reordered, glue("../../pipeline_testing/consolidated_data_samplewise_test/{samples[i]}_consolidated_data.csv"))     
}